# Transformer-Based LLM Chatbot using Hugging Face Transformers and Gradio

**Vishal S**  `24BAD130`

## Part A — Environment Setup and Model Selection

Install the libraries needed for the pretrained model, PyTorch, and Gradio interface.

In [1]:
# Student Roll No: YOUR_ROLL_NUMBER
%pip -q install -U transformers accelerate gradio

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.0.1 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.2.0 which is incompatible.
You should consider upgrading via the 'c:\Users\VISHAL S\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [ ]:
# Student Roll No: YOUR_ROLL_NUMBER
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Select GPU when available; otherwise use CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print("Loading model:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto"
)
model.to(device)
model.eval()

print("Model and tokenizer loaded successfully!")

c:\Users\VISHAL S\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Loading model: Qwen/Qwen2.5-0.5B-Instruct


c:\Users\VISHAL S\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\VISHAL S\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\VISHAL S\AppData\Local\Programs\Python\Python310\lib\site-packages\huggi

In [ ]:
# Student Roll No: YOUR_ROLL_NUMBER
# Simple first test to confirm that the model generates a response

test_messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer clearly and briefly."},
    {"role": "user", "content": "What is artificial intelligence?"}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)
inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
first_response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("Prompt: What is artificial intelligence?")
print("Response:", first_response)

## Part B — Prompt Handling and Response Generation

The function below accepts a prompt, includes recent conversation history, generates a response, checks for empty input, and catches errors.

In [ ]:
# Student Roll No: YOUR_ROLL_NUMBER

SYSTEM_PROMPT = (
    "You are a helpful AI chatbot for a student. "
    "Answer in simple, clear language. If you are unsure, say so."
)

def chatbot_response(message, history=None):
    # Handle empty input
    if not message or not message.strip():
        return "Please enter a message."

    try:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]

        # Include recent conversation history when available
        if history:
            for item in history[-6:]:
                if isinstance(item, dict):
                    role = item.get("role")
                    content = item.get("content", "")
                    if role in ("user", "assistant") and isinstance(content, str):
                        messages.append({"role": role, "content": content})
                elif isinstance(item, (list, tuple)) and len(item) == 2:
                    user_text, assistant_text = item
                    if user_text:
                        messages.append({"role": "user", "content": str(user_text)})
                    if assistant_text:
                        messages.append({"role": "assistant", "content": str(assistant_text)})

        messages.append({"role": "user", "content": message.strip()})

        # Format messages using the model's chat template
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
            truncation=True,
            max_length=2048
        )
        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=120,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode only the newly generated tokens, not the prompt
        prompt_length = inputs["input_ids"].shape[-1]
        response = tokenizer.decode(
            output_ids[0][prompt_length:],
            skip_special_tokens=True
        ).strip()

        if not response:
            return "Sorry, I could not generate a response. Please try another prompt."

        return response

    except Exception as error:
        print("Chatbot error:", error)
        return "Sorry, an error occurred while generating the response. Please try again."

print("Chatbot response function is ready.")

In [ ]:
# Student Roll No: YOUR_ROLL_NUMBER
# Test at least five different prompts

test_prompts = [
    "What is artificial intelligence?",
    "Explain Python in simple words.",
    "What is a Transformer in machine learning?",
    "Give me three tips to prepare for an exam.",
    "Write a short motivational message for a student."
]

for number, prompt in enumerate(test_prompts, start=1):
    answer = chatbot_response(prompt, history=[])
    print(f"TEST {number}")
    print("Prompt:", prompt)
    print("Response:", answer)
    print("-" * 70)

In [ ]:
# Student Roll No: YOUR_ROLL_NUMBER
# Test empty input and a follow-up conversation

print("Empty input test:")
print(chatbot_response("   "))

sample_history = [
    {"role": "user", "content": "My name is Sam."},
    {"role": "assistant", "content": "Nice to meet you, Sam."}
]

print("\nConversation history test:")
print("Prompt: What is my name?")
print("Response:", chatbot_response("What is my name?", history=sample_history))

## Part C — Build the Chatbot Interface

Run the following cell to create an interactive Gradio chatbot. It displays messages, passes conversation history to the response function, and provides a clear-chat control.

In [ ]:
# Student Roll No: YOUR_ROLL_NUMBER

demo = gr.ChatInterface(
    fn=chatbot_response,
    type="messages",
    title="Transformer-Based AI Chatbot",
    description=(
        "A simple chatbot powered by Hugging Face Transformers and "
        "Qwen2.5-0.5B-Instruct. Enter a prompt below to begin."
    ),
    chatbot=gr.Chatbot(type="messages", height=420),
    textbox=gr.Textbox(
        placeholder="Type your message and press Enter...",
        label="Your message"
    ),
    examples=[
        "What is artificial intelligence?",
        "Explain Python in simple words.",
        "What is a Transformer model?",
        "Give me three study tips."
    ]
)

print("Gradio interface created. Launch it in the next cell.")

In [ ]:
# Student Roll No: YOUR_ROLL_NUMBER
# In Google Colab, share=True provides a temporary public demo URL.
# Do not use share=True for private or sensitive conversations.

demo.launch(share=True)

## Part D — Testing, Evaluation, and Deployment

Use the next cell to run a small functional test. Manually review each answer for relevance, correctness, readability, and consistency. A response being generated does not guarantee that it is factually correct.

In [ ]:
# Student Roll No: YOUR_ROLL_NUMBER

evaluation_cases = {
    "Factual": "What is machine learning?",
    "Conversational": "Hello! How are you?",
    "Task-oriented": "List three ways to manage study time."
}

for category, prompt in evaluation_cases.items():
    answer = chatbot_response(prompt, history=[])
    print("Category:", category)
    print("Prompt:", prompt)
    print("Response:", answer)
    print("Manual review needed: Check accuracy and relevance.")
    print("-" * 70)

In [ ]:
# Student Roll No: YOUR_ROLL_NUMBER
# Create a requirements.txt file for local use or deployment

requirements_text = """torch
transformers
accelerate
gradio
"""

with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write(requirements_text)

print("requirements.txt created.")
print(requirements_text)

## Running locally and deploying online

### Run locally
1. Save the complete chatbot code (model loading, `chatbot_response`, and Gradio interface) in a Python file named `app.py`.
2. Install packages with `pip install -r requirements.txt`.
3. Run `python app.py` and open the local URL printed in the terminal. For local launch, change `demo.launch(share=True)` to `demo.launch()`.

### Deploy with Hugging Face Spaces
1. Create a new Space at https://huggingface.co/spaces and choose **Gradio** as the SDK.
2. Upload `app.py` and `requirements.txt`.
3. Wait for the model and dependencies to load, then test the Space URL.
4. Free hosting can have memory and startup limits; if the model does not fit, use a smaller model or a hosted inference API.

### GitHub safety
This notebook uses a public pretrained model and does not need an API key. If you later use an API, store its key in an environment variable or hosting secrets; never hard-code it or commit it to GitHub.

### Suggested screenshots for the lab record
- Part A: library installation and successful model loading.
- Part B: response function and outputs for five prompts.
- Part C: working Gradio chat interface.
- Part D: evaluation output and public application URL, if deployed.

### Conclusion
A prompt-based chatbot was built using a pretrained Transformer language model from Hugging Face, with response generation, basic error handling, conversation history, a Gradio interface, and functional testing. Model responses can still be inaccurate, repetitive, or biased, so they should be checked before relying on them.

### Documentation
- Hugging Face Transformers: https://huggingface.co/docs/transformers/index
- Chat templates and conversational models: https://huggingface.co/docs/transformers/conversations
- Gradio: https://www.gradio.app/docs/gradio/chatinterface